In [2]:
import pandas as pd
import entsoe
import cdsapi
from dotenv import load_dotenv
import os
import time
import calendar


_ = load_dotenv()

# Data Acquisition
In our first step we need to gather the data we will be working with.
Please execute the cell above this. 
It will import the needed packages and load your API keys into your environment.

----

We will be using two major data sources: the ENTSO-E Transparency Platform [1] for all data regarding the energy markets and the ERA5/Copernicus Dataset [2] for weather data. These datasets are quite extensive, accurate and easily acquired.
We will download the data and store them in a file to be processed and analyzed in later steps.

## ENTSO-E

We are starting with the ENTSO-E Datasets as they are the ones we are primarily trying to analyze.
Since I am based in germany we will only be using the data of Germany and since I want to consider the effect of renewable energy I will also include Denmark.
However the principals laid out in this project should be adaptable to most other european countries.
We will only be using data from the years 2019 until 2025. This includes major market disruptions due to the ukraine war and the COVID-19 Pandemic.
The Datasets we will be using are:
  - Day-Ahead Prices
  - Actual Total Load
  - Aggregated Generation per Type

The reason to choose these is that energy prices are heavily influenced by the _Merit-Order-Effect_ [3].
This model orders the different generators from cheapest running cost to highest running cost.
That is why the Aggregated Generation per Type is interesting to us.
The model then checks what the cheapest set of generators are which will still cover the demand.
That is why the Actual Total Load is interesting.
Finally the price of energy is determined by the running cost of the most expensive generator needed to cover demand.
All other generators are able to sell their 'cheaper' energy at the more expensive price.

In [ ]:
OUTPUT_DIR = 'data/raw/entsoe'
YEARS = range(2019,2025+1)
COUNTRIES = ['DE_LU', '10YDK-1--------W', '10YDK-2--------M']


os.makedirs(OUTPUT_DIR, exist_ok=True)
client = entsoe.EntsoePandasClient(api_key=os.environ['ENTSOE_API_KEY'])

for country in COUNTRIES:
    for year in YEARS:
        start = pd.Timestamp(f'{year}0101', tz='UTC')
        end = pd.Timestamp(f'{year}1231', tz='UTC')
        target = os.path.join(OUTPUT_DIR, f'{country}_Price_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            while True:
                try:
                    print(f'requesting: {country}, {year}, price -> {target}')
                    df = client.query_day_ahead_prices(country, start, end).to_frame(name="price")
                    print('received')
                    df.to_parquet(target)
                    break
                except Exception as err:
                    print(f'Problem: {err}')
                    print('Trying again in 120 s')
                    time.sleep(120)

        target = os.path.join(OUTPUT_DIR, f'{country}_Generation_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            while True:
                try:
                    print(f'requesting: {country}, {year}, generation -> {target}')
                    df = client.query_generation(country, start=start, end=end, nett=False)
                    print('received')
                    df.to_parquet(target)
                    break
                except Exception as err:
                    print(f'Problem: {err}')
                    print('Trying again in 120 s')
                    time.wait(120)

        target = os.path.join(OUTPUT_DIR, f'{country}_Load_{year}.parquet')
        if os.path.exists(target):

            print(f'skipping {target} (already exists)')
        else:
            while True:
                try:
                    print(f'requesting: {country}, {year}, load -> {target}')
                    df = client.query_load(country, start=start, end=end)
                    print('received')
                    df.to_parquet(target)
                    break
                except Exception as err:
                    print(f'Problem: {err}')
                    print('Trying again in 120 s')
                    time.wait(120)

skipping data/raw/entsoe/DE_LU_Price_2019.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Generation_2019.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Load_2019.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Price_2020.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Generation_2020.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Load_2020.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Price_2021.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Generation_2021.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Load_2021.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Price_2022.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Generation_2022.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Load_2022.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Price_2023.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Generation_2023.parquet (already exists)
skipping data/raw/entsoe/DE_LU_Load_2023.

/opt/miniconda3/envs/energy-market-analysis/lib/python3.11/site-packages/pandas/io/parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


received
requesting: 10YDK-1--------W, 2020, price -> data/raw/entsoe/10YDK-1--------W_Price_2020.parquet
received
requesting: 10YDK-1--------W, 2020, generation -> data/raw/entsoe/10YDK-1--------W_Generation_2020.parquet
received
requesting: 10YDK-1--------W, 2020, load -> data/raw/entsoe/10YDK-1--------W_Load_2020.parquet
received
requesting: 10YDK-1--------W, 2021, price -> data/raw/entsoe/10YDK-1--------W_Price_2021.parquet
received
requesting: 10YDK-1--------W, 2021, generation -> data/raw/entsoe/10YDK-1--------W_Generation_2021.parquet
received
requesting: 10YDK-1--------W, 2021, load -> data/raw/entsoe/10YDK-1--------W_Load_2021.parquet


/opt/miniconda3/envs/energy-market-analysis/lib/python3.11/site-packages/pandas/io/parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


received
requesting: 10YDK-1--------W, 2022, price -> data/raw/entsoe/10YDK-1--------W_Price_2022.parquet
received
requesting: 10YDK-1--------W, 2022, generation -> data/raw/entsoe/10YDK-1--------W_Generation_2022.parquet
received
requesting: 10YDK-1--------W, 2022, load -> data/raw/entsoe/10YDK-1--------W_Load_2022.parquet
received
requesting: 10YDK-1--------W, 2023, price -> data/raw/entsoe/10YDK-1--------W_Price_2023.parquet
received
requesting: 10YDK-1--------W, 2023, generation -> data/raw/entsoe/10YDK-1--------W_Generation_2023.parquet
received
requesting: 10YDK-1--------W, 2023, load -> data/raw/entsoe/10YDK-1--------W_Load_2023.parquet
received
requesting: 10YDK-1--------W, 2024, price -> data/raw/entsoe/10YDK-1--------W_Price_2024.parquet
received
requesting: 10YDK-1--------W, 2024, generation -> data/raw/entsoe/10YDK-1--------W_Generation_2024.parquet
received
requesting: 10YDK-1--------W, 2024, load -> data/raw/entsoe/10YDK-1--------W_Load_2024.parquet
received
requesting: 1

/opt/miniconda3/envs/energy-market-analysis/lib/python3.11/site-packages/pandas/io/parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


received
requesting: 10YDK-2--------M, 2022, price -> data/raw/entsoe/10YDK-2--------M_Price_2022.parquet
received
requesting: 10YDK-2--------M, 2022, generation -> data/raw/entsoe/10YDK-2--------M_Generation_2022.parquet
received
requesting: 10YDK-2--------M, 2022, load -> data/raw/entsoe/10YDK-2--------M_Load_2022.parquet
received
requesting: 10YDK-2--------M, 2023, price -> data/raw/entsoe/10YDK-2--------M_Price_2023.parquet
received
requesting: 10YDK-2--------M, 2023, generation -> data/raw/entsoe/10YDK-2--------M_Generation_2023.parquet
received
requesting: 10YDK-2--------M, 2023, load -> data/raw/entsoe/10YDK-2--------M_Load_2023.parquet
received
requesting: 10YDK-2--------M, 2024, price -> data/raw/entsoe/10YDK-2--------M_Price_2024.parquet
received
requesting: 10YDK-2--------M, 2024, generation -> data/raw/entsoe/10YDK-2--------M_Generation_2024.parquet
received
requesting: 10YDK-2--------M, 2024, load -> data/raw/entsoe/10YDK-2--------M_Load_2024.parquet
received
requesting: 1

## Copernicus
Weather data is hugely important for the energy markets,
it directly influences both sides of the Merit-Order-Model: the generation capacity of wind, solar and water energy directly correspond to the weather you are having (or had).
But also the energy consumption changes dramatically with the weather. If it is cold people will be consuming more energy to heat.
While on particularly hot days people might be more prone to turning on air conditioning.

In particularly extreme cases weather can even produce outages and disrupt the entire energy network.

_Note: I am unsure of how much industrial energy consumption varies with the weather._

In [ ]:
OUTPUT_DIR = 'data/raw/era5'
DATASET = 'reanalysis-era5-single-levels'

# rough bounding boxes
COUNTRY_AREAS = { # [North, West, South, East]
    'germany': [55.1, 5.8, 47.2, 15.1],
    'luxembourg': [50.2, 5.7, 49.4, 6.5],
    'denmark': [57.8, 8.0, 54.5, 15.2],
}

VARIABLES = [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '100m_u_component_of_wind',
    '100m_v_component_of_wind',
    '2m_temperature',
    'surface_solar_radiation_downwards',
]

YEARS = range(2019,2025+1)
MONTHS = range(1, 13)
ALL_HOURS = [f'{h:02d}:00' for h in range(24)]


def build_request(area: list[float], year: int, month: int) -> dict:
    days_per_month = {
        month: [f'{d:02d}' for d in range(1, calendar.monthrange(year, month)[1] + 1)]
        for month in range(1, 13)
    }
    all_days = [f'{d:02d}' for d in range(1, 32)]

    return {
        'product_type': ['reanalysis'],
        'variable': VARIABLES,
        'year': [str(year)],
        'month': [f'{month:02d}'],
        'day': all_days,
        'time': ALL_HOURS,
        'area': area,  # [North, West, South, East]
        'data_format': 'netcdf',
        'download_format': 'unarchived',
    }


def download_all() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    client = cdsapi.Client(url=os.environ['CDS_API_URL'], key= os.environ['CDS_API_KEY'])

    for country, area in COUNTRY_AREAS.items():
        for year in YEARS:
            for month in MONTHS:
                target = os.path.join(OUTPUT_DIR, f'era5_{country}_{year}_{month}.nc')
                if os.path.exists(target):
                    print(f'skipping {target} (already exists)')
                    continue
    
                request = build_request(area, year, month)
                print(f'requesting: {country}, {year}, {month} -> {target}')
                client.retrieve(DATASET, request, target)


download_all()


## Sources
[1] ENTSO-E, "ENTSO-E Transparency Platform", 2026. [Online]. Available: https://transparency.entsoe.eu/. [Accessed: 10.09.2026].
  
[2] H. Hersbach et al., "ERA5 hourly data on single levels from 1940 to present",
    Copernicus Climate Change Service (C3S) Climate Data Store (CDS), 2018.
    [Online]. Available: https://doi.org/10.24381/cds.adbb2d47.
    [Accessed: 13.09.2026].
    
[3] F. Sensfuß, M. Ragwitz, and M. Genoese, "The merit-order effect: A detailed
    analysis of the price effect of renewable electricity generation on spot
    market prices in Germany", Energy Policy, vol. 36, no. 8, pp. 3076–3084, 2008.